<table style="width:100%">
<tr>
<td style="vertical-align:middle; text-align:left;">
<font size="2">
<a href="http://mng.bz/orYv">Build a Large Language Model From Scratch</a> 책의 보조 코드 by <a href="https://sebastianraschka.com">Sebastian Raschka</a><br>
<br>코드 저장소: <a href="https://github.com/rasbt/LLMs-from-scratch">https://github.com/rasbt/LLMs-from-scratch</a>
</font>
</td>
<td style="vertical-align:middle; text-align:left;">
<a href="http://mng.bz/orYv"><img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/cover-small.webp" width="100px"></a>
</td>
</tr>
</table>

# 5장: 비라벨 데이터로 사전학습(Pretraining on Unlabeled Data)

In [ ]:
from importlib.metadata import version

pkgs = ["matplotlib", 
        "numpy", 
        "tiktoken", 
        "torch",
        "tensorflow" # OpenAI의 사전학습된 가중치를 위해 필요
       ]
for p in pkgs:
    print(f"{p} version: {version(p)}")

- 이 장에서는 LLM을 사전학습하기 위한 훈련 루프와 기본적인 모델 평가 코드를 구현합니다
- 이 장의 마지막에서는 OpenAI에서 공개적으로 제공하는 사전학습된 가중치를 우리 모델에 로드하는 방법도 다룹니다

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch05_compressed/chapter-overview.webp" width=500px>

- 이 장에서 다루는 주제는 아래와 같습니다

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch05_compressed/mental-model--0.webp" width=400px>

## 5.1 생성형 텍스트 모델 평가하기(Evaluating generative text models)

- 이 섹션에서는 이전 장의 코드를 사용하여 GPT 모델을 초기화하는 간단한 요약부터 시작합니다
- 그다음 LLM을 위한 기본적인 평가 지표에 대해 논의합니다
- 마지막으로 이 섹션에서는 이러한 평가 지표를 훈련 데이터셋과 검증 데이터셋에 적용합니다

### 5.1.1 GPT를 사용하여 텍스트 생성하기(Using GPT to generate text)

- 이전 장의 코드를 사용하여 GPT 모델을 초기화합니다

In [ ]:
import torch
from previous_chapters import GPTModel
# `previous_chapters.py` 파일이 로컬에서 사용할 수 없는 경우,
# `llms-from-scratch` PyPI 패키지에서 import할 수 있습니다.
# 자세한 내용은 https://github.com/rasbt/LLMs-from-scratch/tree/main/pkg 를 참조하세요
# 예:
# from llms_from_scratch.ch04 import GPTModel

GPT_CONFIG_124M = {
    "vocab_size": 50257,   # 어휘 크기
    "context_length": 256, # 축소된 컨텍스트 길이 (원래: 1024)
    "emb_dim": 768,        # 임베딩 차원
    "n_heads": 12,         # 어텐션 헤드 수
    "n_layers": 12,        # 레이어 수
    "drop_rate": 0.1,      # 드롭아웃 비율
    "qkv_bias": False      # Query-key-value 바이어스
}

torch.manual_seed(123)
model = GPTModel(GPT_CONFIG_124M)
model.eval();  # 추론 중 드롭아웃 비활성화

- 위에서는 0.1의 드롭아웃을 사용했지만, 요즘에는 LLM을 드롭아웃 없이 훈련하는 것이 상대적으로 일반적입니다
- 현대의 LLM들은 또한 쿼리, 키, 값 행렬을 위한 `nn.Linear` 레이어에서 바이어스 벡터를 사용하지 않습니다(초기 GPT 모델들과 달리). 이는 `"qkv_bias": False`로 설정하여 달성됩니다
- 모델 훈련을 위한 계산 자원 요구사항을 줄이기 위해 컨텍스트 길이(`context_length`)를 256 토큰으로만 줄였습니다. 원래 1억 2400만 개 매개변수 GPT-2 모델은 1024 토큰을 사용했습니다
  - 이는 더 많은 독자들이 노트북 컴퓨터에서 코드 예제를 따라하고 실행할 수 있도록 하기 위함입니다
  - 그러나 `context_length`를 1024 토큰으로 늘리는 것은 자유롭게 하셔도 됩니다(이는 코드 변경을 필요로 하지 않습니다)
  - 나중에 1024 `context_length`를 가진 모델도 사전학습된 가중치에서 로드할 것입니다

- 다음으로, 이전 장에서 사용한 `generate_text_simple` 함수를 사용하여 텍스트를 생성합니다
- 또한 이 장 전체에서 사용할 토큰과 텍스트 표현 간의 변환을 위한 두 가지 편의 함수인 `text_to_token_ids`와 `token_ids_to_text`를 정의합니다

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch05_compressed/gpt-process.webp" width=500px>

In [ ]:
import tiktoken
from previous_chapters import generate_text_simple

# 또는:
# from llms_from_scratch.ch04 import generate_text_simple

def text_to_token_ids(text, tokenizer):
    encoded = tokenizer.encode(text, allowed_special={'<|endoftext|>'})
    encoded_tensor = torch.tensor(encoded).unsqueeze(0) # 배치 차원 추가
    return encoded_tensor

def token_ids_to_text(token_ids, tokenizer):
    flat = token_ids.squeeze(0) # 배치 차원 제거
    return tokenizer.decode(flat.tolist())

start_context = "Every effort moves you"
tokenizer = tiktoken.get_encoding("gpt2")

token_ids = generate_text_simple(
    model=model,
    idx=text_to_token_ids(start_context, tokenizer),
    max_new_tokens=10,
    context_size=GPT_CONFIG_124M["context_length"]
)

print("출력 텍스트:\n", token_ids_to_text(token_ids, tokenizer))

- 위에서 볼 수 있듯이, 모델은 아직 훈련되지 않았기 때문에 좋은 텍스트를 생성하지 못합니다
- "좋은 텍스트"가 무엇인지를 숫자 형태로 측정하거나 포착하여 훈련 중에 추적하려면 어떻게 해야 할까요?
- 다음 하위 섹션에서는 훈련 진행 상황을 측정하는 데 사용할 수 있는 생성된 출력에 대한 손실 지표를 계산하는 메트릭을 소개합니다
- LLM 미세조정에 관한 다음 장들에서는 모델 품질을 측정하는 추가적인 방법들도 소개할 것입니다

<br>

### 5.1.2 텍스트 생성 손실 계산하기: 교차 엔트로피와 퍼플렉시티(Calculating the text generation loss: cross-entropy and perplexity)

- 2개의 훈련 예제(행)에 대한 토큰 ID가 포함된 `inputs` 텐서가 있다고 가정해보겠습니다
- `inputs`에 해당하여, `targets`에는 모델이 생성하기를 원하는 목표 토큰 ID가 포함되어 있습니다
- `targets`는 2장에서 데이터로더를 구현할 때 설명한 것처럼 `inputs`를 1 위치만큼 이동한 것임을 주목하세요

In [ ]:
inputs = torch.tensor([[16833, 3626, 6100],   # ["every effort moves",
                       [40,    1107, 588]])   #  "I really like"]

targets = torch.tensor([[3626, 6100, 345  ],  # [" effort moves you",
                        [1107,  588, 11311]]) #  " really like chocolate"]

- `inputs`를 모델에 넣으면, 각각 3개의 토큰으로 구성된 2개의 입력 예제에 대한 로짓 벡터를 얻습니다
- 각 토큰은 어휘 크기에 해당하는 50,257차원 벡터입니다
- 소프트맥스 함수를 적용하면 로짓 텐서를 확률 점수가 포함된 동일한 차원의 텐서로 변환할 수 있습니다

In [ ]:
with torch.no_grad():
    logits = model(inputs)

probas = torch.softmax(logits, dim=-1) # 어휘의 각 토큰의 확률
print(probas.shape) # 형태: (batch_size, num_tokens, vocab_size)

- 아래 그림은 설명 목적으로 매우 작은 어휘를 사용하여 확률 점수를 다시 텍스트로 변환하는 방법을 보여줍니다. 이는 이전 장의 마지막 부분에서 논의한 내용입니다

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch05_compressed/proba-to-text.webp" width=500px>

- 이전 장에서 논의한 것처럼, `argmax` 함수를 적용하여 확률 점수를 예측 토큰 ID로 변환할 수 있습니다
- 위의 소프트맥스 함수는 각 토큰에 대해 50,257차원 벡터를 생성했습니다. `argmax` 함수는 이 벡터에서 가장 높은 확률 점수의 위치를 반환하며, 이는 해당 토큰의 예측 토큰 ID입니다

- 각각 3개의 토큰을 가진 2개의 입력 배치가 있으므로, 2 x 3 개의 예측 토큰 ID를 얻습니다:

In [ ]:
token_ids = torch.argmax(probas, dim=-1, keepdim=True)
print("토큰 ID:\n", token_ids)

- 이러한 토큰들을 디코딩하면, 모델이 예측하기를 원하는 토큰, 즉 목표 토큰과 상당히 다르다는 것을 알 수 있습니다:

In [ ]:
print(f"목표 배치 1: {token_ids_to_text(targets[0], tokenizer)}")
print(f"출력 배치 1: {token_ids_to_text(token_ids[0].flatten(), tokenizer)}")

- 이는 모델이 아직 훈련되지 않았기 때문입니다
- 모델을 훈련하려면 모델이 올바른 예측(목표)에서 얼마나 멀리 떨어져 있는지를 알아야 합니다

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch05_compressed/proba-index.webp" width=500px>

- 목표 인덱스에 해당하는 토큰 확률은 다음과 같습니다:

In [ ]:
text_idx = 0
target_probas_1 = probas[text_idx, [0, 1, 2], targets[text_idx]]
print("텍스트 1:", target_probas_1)

text_idx = 1
target_probas_2 = probas[text_idx, [0, 1, 2], targets[text_idx]]
print("텍스트 2:", target_probas_2)

- 우리는 이 모든 값들을 최대화하여 1에 가까운 확률로 만들고자 합니다
- 수학적 최적화에서는 확률 점수 자체보다 확률 점수의 로그를 최대화하는 것이 더 쉽습니다. 이는 이 책의 범위를 벗어나지만, 더 자세한 내용은 제가 녹음한 강의를 참조하세요: [L8.2 Logistic Regression Loss Function](https://www.youtube.com/watch?v=GxJe0DZvydM)

In [ ]:
# 모든 토큰 확률의 로그를 계산
log_probas = torch.log(torch.cat((target_probas_1, target_probas_2)))
print(log_probas)

- 다음으로, 평균 로그 확률을 계산합니다:

In [ ]:
# 각 토큰의 평균 확률을 계산
avg_log_probas = torch.mean(log_probas)
print(avg_log_probas)

- 목표는 모델 가중치를 최적화하여 이 평균 로그 확률을 가능한 한 크게 만드는 것입니다
- 로그 때문에 가능한 가장 큰 값은 0이며, 현재 우리는 0에서 멀리 떨어져 있습니다

- 딥러닝에서는 평균 로그 확률을 최대화하는 대신, *음의* 평균 로그 확률 값을 최소화하는 것이 표준 관례입니다. 우리의 경우 -10.7722를 최대화하여 0에 접근시키는 대신, 딥러닝에서는 10.7722를 최소화하여 0에 접근시킵니다
- -10.7722의 음수 값인 10.7722는 딥러닝에서 교차 엔트로피 손실(cross-entropy loss)이라고 불립니다

In [ ]:
neg_avg_log_probas = avg_log_probas * -1
print(neg_avg_log_probas)

- PyTorch는 이미 이전 단계들을 수행하는 `cross_entropy` 함수를 구현하고 있습니다

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch05_compressed/cross-entropy.webp?123" width=400px>

- `cross_entropy` 함수를 적용하기 전에, 로짓과 목표의 형태를 확인해보겠습니다

In [ ]:
# 로짓의 형태는 (batch_size, num_tokens, vocab_size)
print("로짓 형태:", logits.shape)

# 목표의 형태는 (batch_size, num_tokens)
print("목표 형태:", targets.shape)

- PyTorch의 `cross_entropy` 함수를 위해서는 배치 차원에 걸쳐 결합하여 이러한 텐서들을 평탄화하고자 합니다:

In [ ]:
logits_flat = logits.flatten(0, 1)
targets_flat = targets.flatten()

print("평탄화된 로짓:", logits_flat.shape)
print("평탄화된 목표:", targets_flat.shape)

- 목표는 토큰 ID이며, 이는 또한 최대화하고자 하는 로짓 텐서의 인덱스 위치를 나타냅니다
- PyTorch의 `cross_entropy` 함수는 최대화되어야 하는 로짓의 토큰 인덱스에 대해 소프트맥스와 로그 확률 계산을 내부적으로 자동으로 처리합니다

In [ ]:
loss = torch.nn.functional.cross_entropy(logits_flat, targets_flat)
print(loss)

- 교차 엔트로피 손실과 관련된 개념은 LLM의 퍼플렉시티(perplexity)입니다
- 퍼플렉시티는 단순히 교차 엔트로피 손실의 지수입니다

In [ ]:
perplexity = torch.exp(loss)
print(perplexity)

- 퍼플렉시티는 모델이 각 단계에서 불확실해하는 효과적인 어휘 크기로 이해될 수 있기 때문에 더 해석하기 쉬운 것으로 여겨집니다(위 예제에서는 48,725개의 단어나 토큰)
- 다시 말해, 퍼플렉시티는 모델이 예측한 확률 분포가 데이터셋에서 단어의 실제 분포와 얼마나 잘 일치하는지에 대한 측정을 제공합니다
- 손실과 마찬가지로, 낮은 퍼플렉시티는 모델 예측이 실제 분포에 더 가깝다는 것을 나타냅니다

### 5.1.3 훈련 및 검증 세트 손실 계산하기(Calculating the training and validation set losses)

- LLM 훈련을 위해 상대적으로 작은 데이터셋을 사용합니다(실제로는 하나의 짧은 소설만)
- 이유는 다음과 같습니다:
  - 적절한 GPU 없이도 노트북 컴퓨터에서 몇 분 만에 코드 예제를 실행할 수 있습니다
  - 훈련이 상대적으로 빠르게 완료되어(몇 주가 아닌 몇 분) 교육 목적에 적합합니다
  - 사용권을 위반하거나 저장소 크기를 늘리지 않고 이 GitHub 저장소에 포함될 수 있는 공개 도메인의 텍스트를 사용합니다

- 예를 들어, Llama 2 7B는 2조 토큰에서 A100 GPU에서 184,320 GPU 시간이 필요했습니다
  - 이 글을 쓰는 시점에서 AWS의 8xA100 클라우드 서버의 시간당 비용은 약 $30입니다
  - 따라서 간단한 계산으로, 이 LLM을 훈련하는 데 184,320 / 8 * $30 = $690,000이 소요될 것입니다

- 아래에서는 2장에서 사용한 것과 동일한 데이터셋을 사용합니다

In [ ]:
import os
import urllib.request

file_path = "the-verdict.txt"
url = "https://raw.githubusercontent.com/rasbt/LLMs-from-scratch/main/ch02/01_main-chapter-code/the-verdict.txt"

if not os.path.exists(file_path):
    with urllib.request.urlopen(url) as response:
        text_data = response.read().decode('utf-8')
    with open(file_path, "w", encoding="utf-8") as file:
        file.write(text_data)
else:
    with open(file_path, "r", encoding="utf-8") as file:
        text_data = file.read()

- 텍스트가 올바르게 로드되었는지 확인하기 위해 처음과 마지막 99자를 출력합니다

In [ ]:
# 처음 99자
print(text_data[:99])

In [ ]:
# 마지막 99자
print(text_data[-99:])

In [ ]:
total_characters = len(text_data)
total_tokens = len(tokenizer.encode(text_data))

print("문자 수:", total_characters)
print("토큰 수:", total_tokens)

- 5,145개 토큰으로 텍스트는 LLM 훈련에는 매우 짧지만, 다시 말하지만 이는 교육 목적입니다(나중에 사전학습된 가중치도 로드할 것입니다)

- 다음으로, 데이터셋을 훈련 세트와 검증 세트로 나누고 2장의 데이터 로더를 사용하여 LLM 훈련을 위한 배치를 준비합니다
- 시각화 목적으로 아래 그림은 `max_length=6`을 가정하지만, 훈련 로더의 경우 `max_length`를 LLM이 지원하는 컨텍스트 길이와 같게 설정합니다
- 아래 그림은 단순화를 위해 입력 토큰만 보여줍니다
    - LLM을 텍스트의 다음 단어를 예측하도록 훈련하므로, 목표는 이러한 입력과 동일하게 보이지만 목표는 1 위치만큼 이동되어 있습니다

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch05_compressed/batching.webp" width=500px>

In [ ]:
from previous_chapters import create_dataloader_v1
# 또는:
# from llms_from_scratch.ch02 import create_dataloader_v1

# 훈련/검증 비율
train_ratio = 0.90
split_idx = int(train_ratio * len(text_data))
train_data = text_data[:split_idx]
val_data = text_data[split_idx:]


torch.manual_seed(123)

train_loader = create_dataloader_v1(
    train_data,
    batch_size=2,
    max_length=GPT_CONFIG_124M["context_length"],
    stride=GPT_CONFIG_124M["context_length"],
    drop_last=True,
    shuffle=True,
    num_workers=0
)

val_loader = create_dataloader_v1(
    val_data,
    batch_size=2,
    max_length=GPT_CONFIG_124M["context_length"],
    stride=GPT_CONFIG_124M["context_length"],
    drop_last=False,
    shuffle=False,
    num_workers=0
)

In [ ]:
# 정상성 확인

if total_tokens * (train_ratio) < GPT_CONFIG_124M["context_length"]:
    print("훈련 로더에 토큰이 충분하지 않습니다. "
          "`GPT_CONFIG_124M['context_length']`를 낮추거나 "
          "`training_ratio`를 높여보세요")

if total_tokens * (1-train_ratio) < GPT_CONFIG_124M["context_length"]:
    print("검증 로더에 토큰이 충분하지 않습니다. "
          "`GPT_CONFIG_124M['context_length']`를 낮추거나 "
          "`training_ratio`를 낮춰보세요")

- 계산 자원 요구사항을 줄이기 위해 상대적으로 작은 배치 크기를 사용하며, 데이터셋이 처음부터 매우 작기 때문입니다
- 예를 들어, Llama 2 7B는 1024의 배치 크기로 훈련되었습니다

- 데이터가 올바르게 로드되었는지에 대한 선택적 확인:

In [ ]:
print("훈련 로더:")
for x, y in train_loader:
    print(x.shape, y.shape)

print("\n검증 로더:")
for x, y in val_loader:
    print(x.shape, y.shape)

- 토큰 크기가 예상 범위에 있는지에 대한 또 다른 선택적 확인:

In [ ]:
train_tokens = 0
for input_batch, target_batch in train_loader:
    train_tokens += input_batch.numel()

val_tokens = 0
for input_batch, target_batch in val_loader:
    val_tokens += input_batch.numel()

print("훈련 토큰:", train_tokens)
print("검증 토큰:", val_tokens)
print("전체 토큰:", train_tokens + val_tokens)

- 다음으로, 주어진 배치의 교차 엔트로피 손실을 계산하는 유틸리티 함수를 구현합니다
- 추가로, 데이터 로더에서 사용자가 지정한 수의 배치에 대한 손실을 계산하는 두 번째 유틸리티 함수를 구현합니다

In [ ]:
def calc_loss_batch(input_batch, target_batch, model, device):
    input_batch, target_batch = input_batch.to(device), target_batch.to(device)
    logits = model(input_batch)
    loss = torch.nn.functional.cross_entropy(logits.flatten(0, 1), target_batch.flatten())
    return loss


def calc_loss_loader(data_loader, model, device, num_batches=None):
    total_loss = 0.
    if len(data_loader) == 0:
        return float("nan")
    elif num_batches is None:
        num_batches = len(data_loader)
    else:
        # num_batches가 데이터 로더의 배치 수를 초과하는 경우
        # 데이터 로더의 전체 배치 수에 맞춰 배치 수를 줄입니다
        num_batches = min(num_batches, len(data_loader))
    for i, (input_batch, target_batch) in enumerate(data_loader):
        if i < num_batches:
            loss = calc_loss_batch(input_batch, target_batch, model, device)
            total_loss += loss.item()
        else:
            break
    return total_loss / num_batches

- CUDA 지원 GPU가 있는 머신을 사용하는 경우, 코드를 변경하지 않고도 LLM이 GPU에서 훈련됩니다
- `device` 설정을 통해 데이터가 LLM 모델과 동일한 장치에 로드되도록 보장합니다

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# 참고:
# 다음 줄들의 주석을 해제하면 Apple Silicon 칩에서 코드를 실행할 수 있습니다(해당하는 경우).
# 이는 Apple CPU에서보다 약 2배 빠릅니다(M3 MacBook Air에서 측정).
# 그러나 결과 손실 값들이 약간 다를 수 있습니다.

#if torch.cuda.is_available():
#    device = torch.device("cuda")
#elif torch.backends.mps.is_available():
#    device = torch.device("mps")
#else:
#    device = torch.device("cpu")
#
# print(f"Using {device} device.")


model.to(device) # nn.Module 클래스의 경우 model = model.to(device) 할당 불필요


torch.manual_seed(123) # 데이터 로더의 셔플링으로 인한 재현가능성을 위해

with torch.no_grad(): # 아직 훈련하지 않으므로 효율성을 위해 그래디언트 추적 비활성화
    train_loss = calc_loss_loader(train_loader, model, device)
    val_loss = calc_loss_loader(val_loader, model, device)

print("훈련 손실:", train_loss)
print("검증 손실:", val_loss)

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch05_compressed/mental-model-1.webp" width=400px>